# Model 2 — WavLM-FT+Aug

Reviewer-facing reproduction notebook for the reported BNSER-WavLM configuration.

In [ ]:
import json
import sys
from pathlib import Path

import torch

# Locate experiments/training without embedding a private local path.
TRAINING_ROOT = None
start = Path.cwd().resolve()
for base in [start, *start.parents]:
    for candidate in (base / "experiments" / "training", base):
        if (candidate / "common").is_dir() and (candidate / "configs").is_dir():
            TRAINING_ROOT = candidate
            break
    if TRAINING_ROOT is not None:
        break

if TRAINING_ROOT is None:
    raise FileNotFoundError("Could not locate experiments/training/common and configs.")

if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))

import yaml

from common.data import (
    create_dataloaders,
    create_feature_extractor,
    load_predefined_splits,
    write_split_manifests,
)
from common.evaluation import predict, save_evaluation_outputs
from common.model import build_model, make_loss, model_parameter_summary
from common.training import train_model
from common.utils import resolve_data_root, save_json, set_seed

CONFIG_PATH = TRAINING_ROOT / "configs" / "model2_wavlm_aug.yaml"
OUTPUT_DIR = TRAINING_ROOT / "outputs" / "model2_wavlm_aug"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    CONFIG = yaml.safe_load(file)

DATA_ROOT = resolve_data_root()
set_seed(int(CONFIG["training"]["seed"]))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Configuration:", CONFIG_PATH)
print("Dataset:", DATA_ROOT)
print("Device:", DEVICE)

## Fixed dataset and preprocessing

The predefined speaker-independent `Train/Val/Test` folders are used directly. Audio is loaded at 16 kHz, converted to mono, fixed to four seconds, and normalized by the Wav2Vec2 feature extractor.

In [ ]:
SPLIT_NAMES = CONFIG["data"]["split_names"]
SAMPLE_RATE = int(CONFIG["data"]["sample_rate"])
TARGET_LENGTH = int(
    SAMPLE_RATE * CONFIG["data"]["duration_seconds"]
)
BATCH_SIZE = int(CONFIG["training"]["batch_size"])

splits = load_predefined_splits(DATA_ROOT, SPLIT_NAMES)
observed_sizes = {name: len(records) for name, records in splits.items()}
expected_sizes = {"train": 1610, "validation": 345, "test": 345}

print("Split sizes:", observed_sizes)

if observed_sizes != expected_sizes:
    raise ValueError(
        f"Unexpected split sizes: {observed_sizes}; "
        f"expected {expected_sizes}"
    )

feature_extractor = create_feature_extractor(
    CONFIG["model"]["name"],
    TARGET_LENGTH,
)

## Training-only augmentation

Model 2 applies the controlled augmentation only to the training dataset. Validation and test inputs are unaugmented.

In [ ]:
augmentation = None

if CONFIG["augmentation"]["enabled"]:
    from common.augmentation import ControlledAugmentation

    augmentation_config = CONFIG["augmentation"]
    augmentation = ControlledAugmentation(
        probability=float(augmentation_config["probability"]),
        noise_probability=float(augmentation_config["noise_probability"]),
        pitch_probability=float(augmentation_config["pitch_probability"]),
        time_stretch_probability=float(
            augmentation_config["time_stretch_probability"]
        ),
        noise_level_min=float(augmentation_config["noise_level_min"]),
        noise_level_max=float(augmentation_config["noise_level_max"]),
        pitch_steps_min=float(augmentation_config["pitch_steps_min"]),
        pitch_steps_max=float(augmentation_config["pitch_steps_max"]),
        stretch_rate_min=float(augmentation_config["stretch_rate_min"]),
        stretch_rate_max=float(augmentation_config["stretch_rate_max"]),
        sample_rate=SAMPLE_RATE,
    )

loaders = create_dataloaders(
    splits,
    feature_extractor,
    SAMPLE_RATE,
    TARGET_LENGTH,
    BATCH_SIZE,
    augmentation=augmentation,
    train_crop_mode=CONFIG["data"]["train_crop_mode"],
    eval_crop_mode=CONFIG["data"]["eval_crop_mode"],
)

print("Training batches:", len(loaders["train"]))
print("Validation batches:", len(loaders["validation"]))
print("Test batches:", len(loaders["test"]))

## WavLM-Large model and loss

Model 2 fully fine-tunes all WavLM-Large layers and uses standard cross-entropy.

In [ ]:
model = build_model(
    CONFIG["model"]["name"],
    int(CONFIG["model"]["num_labels"]),
    int(CONFIG["model"]["freeze_first_n_layers"]),
).to(DEVICE)

parameter_summary = model_parameter_summary(model)
print(json.dumps(parameter_summary, indent=2))

criterion, class_weights = make_loss(
    CONFIG["loss"]["class_weights"],
    DEVICE,
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(CONFIG["training"]["learning_rate"]),
)

print("Class weights:", class_weights.detach().cpu().tolist())

## Training

AdamW is used with maximum 50 epochs and five-epoch early stopping patience. Model 3 uses four-step gradient accumulation with physical batch size 4 (effective batch size 16).

In [ ]:
write_split_manifests(
    splits,
    DATA_ROOT,
    OUTPUT_DIR / "manifests",
)
save_json(CONFIG, OUTPUT_DIR / "config.json")
save_json(
    parameter_summary,
    OUTPUT_DIR / "parameter_summary.json",
)

train_summary, history = train_model(
    model=model,
    train_loader=loaders["train"],
    validation_loader=loaders["validation"],
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    max_epochs=int(CONFIG["training"]["max_epochs"]),
    patience=int(CONFIG["training"]["early_stopping_patience"]),
    accumulation_steps=int(
        CONFIG["training"]["gradient_accumulation_steps"]
    ),
    output_dir=OUTPUT_DIR,
    validation_loss_aggregation=CONFIG["training"][
        "validation_loss_aggregation"
    ],
)

train_summary.update(parameter_summary)
train_summary["device"] = str(DEVICE)
save_json(
    train_summary,
    OUTPUT_DIR / "training_summary.json",
)

print(json.dumps(train_summary, indent=2))

## Fixed test-set evaluation

The best validation-loss checkpoint is loaded before evaluating the fixed 345-sample test set. No augmentation is used during test inference.

In [ ]:
checkpoint = OUTPUT_DIR / "best_model.pt"

if not checkpoint.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found: {checkpoint}"
    )

model.load_state_dict(
    torch.load(checkpoint, map_location=DEVICE)
)
model.eval()

predictions = predict(
    model,
    loaders["test"],
    DEVICE,
)
metrics = save_evaluation_outputs(
    predictions,
    OUTPUT_DIR,
)

print(json.dumps(metrics, indent=2))

## Reproducibility note

Model 2 adds the manuscript-defined training-time augmentation to the full-fine-tuning reference.

The notebook preserves the documented configuration and fixed speaker-independent split. Exact decimal reproduction may vary with software, hardware, and stochastic execution.